# Test of functionalities

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
from src.data import preprocess_data

## Data Preprocessing

Testing Data Preprocessing: Variable selection, Name cleaning, formatting

In [3]:
train, test = preprocess_data("data/L1", "L1")

In [4]:
train

,date,home_team,away_team,home_goals,away_goals,played,season,t
0,2021-08-06,Monaco,Nantes,1,1,True,0,1
1,2021-08-07,Lyon,Brest,1,1,True,0,1
2,2021-08-07,Troyes,Paris SG,1,2,True,0,1
3,2021-08-08,Rennes,Lens,1,1,True,0,1
4,2021-08-08,Bordeaux,Clermont,0,2,True,0,1
...,...,...,...,...,...,...,...,...
1745,2026-10-25,Angers,Lorient,0,0,True,5,181
1746,2026-10-25,Auxerre,Le Mans,1,1,True,5,181
1747,2026-10-25,Marseille,Le Havre,4,3,True,5,181
1748,2026-10-25,Toulouse,Troyes,0,0,True,5,181


In [5]:
df = pd.read_csv("data/train_L1.csv")

In [6]:
df.dtypes

date            str
home_team       str
away_team       str
home_goals    int64
away_goals    int64
played         bool
season        int64
t             int64
dtype: object

In [7]:
test

,date,home_team,away_team,home_goals,away_goals,played,season,t
1678,2026-08-21,Marseille,Strasbourg,4,0,True,5,174
1679,2026-08-22,Lens,Auxerre,5,2,True,5,174
1680,2026-08-22,Le Mans,Brest,2,2,True,5,174
1681,2026-08-22,Nice,Lorient,0,0,True,5,174
1682,2026-08-22,Toulouse,Lyon,0,2,True,5,174
...,...,...,...,...,...,...,...,...
1899,2027-03-20,Lens,Lille,<NA>,<NA>,False,5,198
1900,2027-03-20,Angers,Le Mans,<NA>,<NA>,False,5,198
1901,2027-03-20,Rennes,Paris FC,<NA>,<NA>,False,5,198
1902,2027-03-20,Paris SG,Strasbourg,<NA>,<NA>,False,5,198


## Model instanciation

### Static Double Poisson model

In [8]:
from src.model import StaticDoublePoisson

In [9]:
SPoisson = StaticDoublePoisson()

In [10]:
df = pd.read_csv("data/train_L1.csv")
SPoisson.fit(df)

Optimization terminated successfully    (Exit mode 0)
            Current function value: 2968.519754514572
            Iterations: 43
            Function evaluations: 2412
            Gradient evaluations: 43


In [11]:
# Toulouse VS Strasbourg Simulations
print("---Single Simulation---")
print(SPoisson.predict("Toulouse","Strasbourg"))
print("---Score prediction---")
print(SPoisson.predict_proba("Toulouse","Strasbourg"))
print("---Outcome prediction---")
print(SPoisson.predict_result("Toulouse","Strasbourg"))

---Single Simulation---
{'Toulouse': np.int32(3), 'Strasbourg': np.int32(1)}
---Score prediction---
{'1-1': 0.129, '1-2': 0.086, '1-0': 0.08, '2-1': 0.078, '0-0': 0.07, '0-1': 0.068, '2-2': 0.059, '2-0': 0.058, '0-2': 0.057, '1-3': 0.046, '3-1': 0.04, '3-2': 0.034, '3-0': 0.032, '0-3': 0.025, '4-1': 0.024, '2-3': 0.021, '1-4': 0.017, '3-3': 0.013, '2-4': 0.013, '4-0': 0.011, '4-2': 0.008, '0-4': 0.006, '5-2': 0.004, '5-1': 0.003, '5-0': 0.002, '5-3': 0.002, '5-5': 0.001, '2-7': 0.001, '6-0': 0.001, '4-4': 0.001, '0-6': 0.001, '3-4': 0.001, '1-5': 0.001, '2-5': 0.001, '2-6': 0.001, '6-1': 0.001, '5-4': 0.001, '0-5': 0.001, '6-2': 0.001, '3-5': 0.001}
---Outcome prediction---
{'Toulouse': np.float64(0.406), 'Draw': np.float64(0.255), 'Strasbourg': np.float64(0.339)}


In [21]:
current_season = pd.read_csv("data/test_L1.csv")

In [24]:
standings,sim = SPoisson.predict_season(current_season,dynamic=True)
print(standings)
print(sim)

          team  played  points    gf    ga    gd  rank
0     Paris SG      25      56  55.0  25.0  30.0     1
1       Monaco      25      44  45.0  32.0  13.0     2
2         Lyon      25      44  37.0  24.0  13.0     3
3     Paris FC      25      39  38.0  30.0   8.0     4
4    Marseille      25      39  46.0  40.0   6.0     5
5        Lille      25      38  44.0  31.0  13.0     6
6      Lorient      25      38  36.0  29.0   7.0     7
7         Lens      25      37  39.0  36.0   3.0     8
8     Toulouse      25      36  35.0  28.0   7.0     9
9       Rennes      25      36  38.0  36.0   2.0    10
10        Nice      25      34  27.0  29.0  -2.0    11
11  Strasbourg      25      33  42.0  47.0  -5.0    12
12     Auxerre      25      32  34.0  38.0  -4.0    13
13       Brest      25      28  33.0  33.0   0.0    14
14     Le Mans      25      25  33.0  53.0 -20.0    15
15      Angers      25      23  27.0  42.0 -15.0    16
16    Le Havre      25      17  24.0  43.0 -19.0    17
17      Tr

In [19]:
res = SPoisson.predict_season_proba(current_season,n_simu=10,dynamic=True)
print(res)

            Title (%)  Top 4 (%)  Relegation (%)
team                                            
Paris SG         80.0      100.0             0.0
Lens             10.0       40.0             0.0
Rennes           10.0       40.0             0.0
Auxerre           0.0        0.0             0.0
Angers            0.0        0.0            10.0
 Monaco           0.0        0.0             0.0
Brest             0.0        0.0            10.0
Lille             0.0       50.0             0.0
Lorient           0.0       10.0             0.0
Le Havre          0.0        0.0            60.0
Le Mans           0.0        0.0            20.0
Marseille         0.0       50.0             0.0
Lyon              0.0       20.0             0.0
Nice              0.0       10.0             0.0
Monaco            0.0       40.0             0.0
Paris FC          0.0       30.0             0.0
Strasbourg        0.0        0.0            10.0
Toulouse          0.0       10.0             0.0
Troyes            0.

### Bivariate Dynamic Poisson model